In [1]:
import xarray as xr
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


In [2]:
pp_path ='data/ClimateScenarios/pr/pr_As_merged.nc'
pp = xr.open_dataset(pp_path)

tas_path ='data/ClimateScenarios/tas/tas_As_merged.nc'
tas = xr.open_dataset(tas_path)

In [3]:
# Define baseline period
baseline_period = ('1991-01-01','2019-12-31')

# Define last period
periods = {('2021-01-01','2050-12-31'),
           ('2031-01-01','2060-12-31'),
           ('2041-01-01','2070-12-31'), 
           ('2051-01-01','2080-12-31'),
           ('2061-01-01','2090-12-31'), 
           ('2071-01-01','2099-12-31') }

def anomalyT(row):
        if row["season"] == "Annual":
            return row["tas"] - annual_climatologyT
        
def anomalyPP(row):
        if row["season"] == "Annual":
            return (row["pr"] - annual_climatologyPP)/annual_climatologyPP * 100 # Percentage change for annual


for period_f in periods:    
    period_name = f"{period_f[0][:4]}_{period_f[1][:4]}"
    for rcp in ['rcp26', 'rcp45', 'rcp85']:  # You can add other RCPs if needed
        # --- Prepare temperature data ---
        tas_rcp = tas.sel(rcp_tag=rcp).mean(dim=['rlon','rlat'])
        dataT = tas_rcp['tas'] - 273.15
        dataT = dataT.to_dataframe().reset_index().dropna(subset=["tas"])
        dataT["year"] = dataT["time"].dt.year
        dataT["month"] = dataT["time"].dt.month

        # --- Annual means (one value per year) ---
        annual_meansT = (
            dataT.groupby(["year",'model_tag'])["tas"]
            .mean()
            .reset_index()
        )
        annual_meansT["season"] = "Annual"
        annual_meansT["time"] = pd.to_datetime(annual_meansT["year"].astype(str) + "-01-01")

        starti, endi = baseline_period
        baselineT = annual_meansT.query("time >= @starti and time <= @endi").copy()

        annual_climatologyT = baselineT.query("season == 'Annual'")["tas"].mean()


        # --- Prepare precipitation data ---
        pr_rcp = pp.sel(rcp_tag=rcp).mean(dim=['rlon','rlat'])
        dataPP = pr_rcp['pr'] * 86400  # Convert to mm/day
        dataPP = dataPP.to_dataframe().reset_index().dropna(subset=["pr"])
        dataPP["year"] = dataPP["time"].dt.year
        dataPP["month"] = dataPP["time"].dt.month

        # --- Annual means (one value per year) ---
        annual_meansPP = (
            dataPP.groupby(["year","model_tag"])["pr"]
            .mean()
            .reset_index()
        )
        annual_meansPP["season"] = "Annual"
        annual_meansPP["time"] = pd.to_datetime(annual_meansPP["year"].astype(str) + "-01-01")
        baselinePP = annual_meansPP.query("time >= @starti and time <= @endi").copy()
        annual_climatologyPP = baselinePP.query("season == 'Annual'")["pr"].mean()
            

        # --- Last period anomalies T ---
        start, end = period_f
        data_fT = annual_meansT.query("time >= @start and time <= @end").copy()
        data_fT["tas_change"] = data_fT.apply(anomalyT, axis=1)

        # --- Last period anomalies PP ---
        data_fPP = annual_meansPP.query("time >= @start and time <= @end").copy()
        data_fPP["pr_change"] = data_fPP.apply(anomalyPP, axis=1)

        results = []

        for model_i in data_fT['model_tag'].unique():
            # Temperature changes
            model_dataT = data_fT[data_fT['model_tag'] == model_i]
            mean_tas_change = model_dataT['tas_change'].mean()

            # Precipitation changes
            model_dataPP = data_fPP[data_fPP['model_tag'] == model_i]
            mean_PP_change = model_dataPP['pr_change'].mean()

            # Append to results
            results.append({
                "model_name": model_i,
                "t_change": mean_tas_change,
                "pp_change": mean_PP_change
            })

        # Convert to dataframe
        rcpmeanChanges = pd.DataFrame(results)
        rcpmeanChanges.to_csv(f'methodology_outs/{rcp}_{period_name}_meanChanges_As.csv', index=False)